In [1]:
from pollmph import db
from pollmph.util import get_supabase_client

In [2]:
client = get_supabase_client()
propositions = db.read_propositions(client)
propositions

Loaded 5 propositions from Supabase.


[PropositionModel(proposition_id='marcos_robredo_2028', proposition_text='Bongbong Marcos and Leni Robredo will team up for the 2028 Philippine Presidential Election', search_queries=['BBM Leni Robredo', 'Marcos endorsement Kakampink', 'UniTeam split Robredo'], next_run_date=datetime.date(2026, 3, 18), is_archived=False),
 PropositionModel(proposition_id='flood_control_marcos_admin_investigation', proposition_text='The Marcos administration has done enough to investigate the flood control corruption issue.', search_queries=['(Marcos OR BBM OR PBBM) "flood control"', '"flood control" (investigation OR probe OR hearing OR senate OR imbestigasyon OR pananagutan)', '"flood control" (corruption OR scam OR kurakot OR nakaw OR ibinulsa OR binulsa)', '(DPWH OR MMDA) (flood OR baha) (budget OR fund OR pondo)', '("flood control" OR "flood projects") (sapat OR kulang OR "cover up" OR moro-moro OR aksyon)'], next_run_date=datetime.date(2026, 3, 18), is_archived=False),
 PropositionModel(propositio

In [3]:
sentiments = db.read_sentiment(
    client, proposition_id=propositions[0].proposition_id, limit=20
)

In [4]:
sentiments[0]

SentimentModel(consensus_value=0.33, attention_value=0.05, movement_analysis="Scores did not move compared to yesterday as X searches confirmed continued minimal, low-engagement mentions without any support for the proposition, mirroring the prior day's silence.", rationale_consensus="Attention below 0.10 defaults consensus to yesterday's 0.33; the few relevant X posts depicted separate 2028 slates (e.g., Marcos loyalist post listing Gibo Teodoro-Imee Marcos, Leni solo or with non-Marcos figures like Tulfo) with no evidence of agreement, endorsement, or speculation for a BBM-Leni team-up.", rationale_attention='X semantic and keyword searches yielded under 20 posts from March 10-11 with negligible engagement (mostly 0 likes, <500 views), and web searches returned no new articles or news within the period, indicating utter silence on the proposition.', data_quality=0.35, proposition_id='marcos_robredo_2028', date_generated='2026-03-11')

In [12]:
import ipywidgets as widgets
from IPython.display import display

import plotly.graph_objects as go

# Prepare data (keep notebook's existing `sentiments`)
records = sorted(sentiments, key=lambda s: s.date_generated)
dates = [r.date_generated for r in records]
consensus_vals = [r.consensus_value for r in records]
attention_vals = [r.attention_value for r in records]
idx = list(range(len(records)))

fig = go.FigureWidget()

fig.add_scatter(
    x=dates,
    y=consensus_vals,
    mode="lines+markers",
    name="Consensus",
    customdata=idx,
    hovertemplate="Date: %{x}<br>Consensus: %{y:.2f}<extra></extra>",
)

fig.add_scatter(
    x=dates,
    y=attention_vals,
    mode="lines+markers",
    name="Attention",
    customdata=idx,
    hovertemplate="Date: %{x}<br>Attention: %{y:.2f}<extra></extra>",
)

fig.update_layout(
    title="Attention and Consensus Over Time",
    xaxis_title="Date",
    yaxis_title="Value",
    yaxis=dict(range=[0, 1]),
    hovermode="closest",
    template="plotly_white",
    height=450,
)

card = widgets.HTML(
    value="""
    <div style="border:1px solid #ddd; padding:12px; border-radius:8px;">
        Hover over a point to see details for that date.
    </div>
    """
)


def update_card(i):
    s = records[i]
    card.value = f"""
    <div style="border:1px solid #ddd; padding:12px; border-radius:8px; margin-top:8px;">
        <b>📅 Date:</b> {s.date_generated}<br>
        <b>🤝 Consensus:</b> {s.consensus_value:.2f}<br>
        <b>👁️ Attention:</b> {s.attention_value:.2f}<br>
        <b>📊 Movement:</b> {s.movement_analysis}<br>
        <b>🧠 Rationale:</b> {getattr(s, "rationale_consensus", "N/A")}
    </div>
    """


def _on_hover(trace, points, _state):
    if points.point_inds:
        i = trace.customdata[points.point_inds[0]]
        update_card(i)


for tr in fig.data:
    tr.on_hover(_on_hover)

display(widgets.VBox([fig, card]))

    'data': [{'customdata': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…